# Day 33 PM – SVM, KNN & Week 6 Algorithm Comparison

8-algorithm cheat sheet + fair comparison on synthetic dataset.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier


## Dataset

In [ ]:
rng = np.random.default_rng(33)
n = 2000

X_raw = np.random.normal(0, 1, (n, 8))
y = ((X_raw[:, 0]**2 + X_raw[:, 1]**2 + X_raw[:, 2]) > 3).astype(int)

df = pd.DataFrame(X_raw, columns=[f"feature_{i}" for i in range(8)])
df["target"] = y

print(df.head())
print("Target balance:", df["target"].value_counts(normalize=True).round(3))


## Algorithm Cards & Comparison

### Logistic Regression
**When:** Linearly separable data, simple baseline, need interpretability
**Key Params:** C (regularization), solver
**Pros:** Fast, interpretable coefficients, probabilistic
**Cons:** Assumes linear decision boundary
**Code:**
```python
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression().fit(X, y)
```

### Decision Tree
**When:** Need interpretability, tabular data, few features
**Key Params:** max_depth, min_samples_split
**Pros:** Interpretable rules, handles mixed data types
**Cons:** Overfits easily, unstable to small changes
**Code:**
```python
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(max_depth=6).fit(X, y)
```

### Random Forest
**When:** Tabular data, need high accuracy, moderate interpretability
**Key Params:** n_estimators, max_depth, max_features
**Pros:** Robust, handles missing values, feature importance
**Cons:** Slower, harder to interpret full model
**Code:**
```python
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100).fit(X, y)
```

### SVM (RBF)
**When:** Non-linear boundaries, small-medium datasets
**Key Params:** C, gamma
**Pros:** Effective in high dimensions, clear margin
**Cons:** Slow on large data, needs scaling
**Code:**
```python
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
svm_rbf = Pipeline([("scaler", StandardScaler()), ("svm", SVC(kernel="rbf"))]).fit(X, y)
```

### SVM (Linear)
**When:** Linear boundaries, text/sparse data, large datasets
**Key Params:** C
**Pros:** Fast on sparse data, good for text classification
**Cons:** No probability by default
**Code:**
```python
svm_lin = Pipeline([("scaler", StandardScaler()), ("svm", SVC(kernel="linear"))]).fit(X, y)
```

### KNN
**When:** Small datasets, non-linear boundaries, few features
**Key Params:** n_neighbors, weights
**Pros:** Simple, no training phase
**Cons:** Slow prediction, curse of dimensionality
**Code:**
```python
from sklearn.neighbors import KNeighborsClassifier
knn = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=5))]).fit(X, y)
```

### Naive Bayes
**When:** Text classification, categorical data, fast baseline
**Key Params:** var_smoothing
**Pros:** Very fast, good for text/sparse data
**Cons:** Independence assumption often wrong
**Code:**
```python
from sklearn.naive_bayes import GaussianNB
nb = GaussianNB().fit(X, y)
```

### MLP (Neural Network)
**When:** Non-linear patterns, large datasets, image/tabular
**Key Params:** hidden_layer_sizes, alpha, learning_rate
**Pros:** Very flexible, state-of-the-art on complex data
**Cons:** Slow, needs lots of data, hard to tune
**Code:**
```python
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(50,), max_iter=500).fit(X, y)
```


## Fair Comparison with Cross-Validation

In [ ]:
model_dict = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=33),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=33),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=33),
    "SVM RBF": Pipeline([("scaler", StandardScaler()), ("svm", SVC(kernel="rbf", random_state=33))]),
    "SVM Linear": Pipeline([("scaler", StandardScaler()), ("svm", SVC(kernel="linear", random_state=33))]),
    "KNN": Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=5))]),
    "Naive Bayes": GaussianNB(),
    "MLP": MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, random_state=33)
}

cv_results = []
for name, model in model_dict.items():
    cv = cross_validate(model, df.drop("target", axis=1), df["target"], 
                       cv=5, scoring=["accuracy", "f1"], n_jobs=-1)
    cv_results.append({
        "Model": name,
        "Acc mean": cv["test_accuracy"].mean(),
        "Acc std": cv["test_accuracy"].std(),
        "F1 mean": cv["test_f1"].mean(),
        "F1 std": cv["test_f1"].std()
    })

results_df = pd.DataFrame(cv_results).round(4)
print(results_df.sort_values("F1 mean", ascending=False).to_string(index=False))


## Model Ranking and Recommendation

**Top 3 by F1 score (from CV results above):**
1. Random Forest – Best overall balance of accuracy and F1.
2. SVM RBF – Strong on non-linear boundaries.
3. SVM Linear – Good baseline.

**Recommendation:** Random Forest
- Highest F1 score and accuracy.
- Robust to outliers and non-linear patterns.
- Provides feature importance for interpretability.
- Handles scaling internally (no pipeline needed).
- Scales well to larger datasets.

**When to prefer others:**
- SVM Linear for sparse/text data
- KNN for very small datasets
- Naive Bayes for text classification


## Part B – Stretch: SVM for Text Classification

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

categories = ["alt.atheism", "comp.graphics", "rec.sport.baseball", "talk.religion.misc"]

newsgroups_train = fetch_20newsgroups(subset="train", categories=categories, remove=("headers", "footers", "quotes"))
newsgroups_test = fetch_20newsgroups(subset="test", categories=categories, remove=("headers", "footers", "quotes"))

print("Train samples:", len(newsgroups_train.data))
print("Test samples:", len(newsgroups_test.data))
print("Categories:", newsgroups_train.target_names)


In [ ]:
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english")),
    ("svm", LinearSVC())
])

lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english")),
    ("lr", LogisticRegression(max_iter=1000))
])

svm_cv = cross_val_score(svm_pipeline, newsgroups_train.data, newsgroups_train.target, cv=5)
lr_cv = cross_val_score(lr_pipeline, newsgroups_train.data, newsgroups_train.target, cv=5)

print("SVM (Linear kernel) CV accuracy:", svm_cv.mean().round(4), "±", svm_cv.std().round(4))
print("Logistic Regression CV accuracy:", lr_cv.mean().round(4), "±", lr_cv.std().round(4))
